Tiatoolbox can extract patches in different ways, such as point-based, fixed-window, and variable-window patch extraction. One practical use of these tools is when using deep learning models that cannot accept large images in the input.


### Importing related libraries
We will start by importing some libraries required to run this notebook.

In [ ]:
"""Import modules required to run the Jupyter notebook."""
import os
import sys
import shutil

# Clear logger to use tiatoolbox.logger
import logging

if logging.getLogger().hasHandlers():
    logging.getLogger().handlers.clear()

from pathlib import Path
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.patches as patches

from tiatoolbox import logger
from tiatoolbox.wsicore.wsireader import WSIReader, VirtualWSIReader
from tiatoolbox.tools import patchextraction
from tiatoolbox.utils.misc import imread, read_locations
from tiatoolbox.tools.patchextraction import PatchExtractor
from tiatoolbox.tools import stainnorm
from tiatoolbox import data

import openslide

from typing import Union, Tuple
from skimage import io
from skimage.filters import threshold_otsu
from skimage.transform import resize
from scipy.signal import convolve2d
import cv2
import xml.etree.ElementTree as ET
from shapely.strtree import STRtree
from shapely.geometry import Polygon
from PIL import Image

mpl.rcParams["figure.dpi"] = 300  # for high resolution figure in notebook
mpl.rcParams["figure.facecolor"] = "white"  # To make sure text is visible in dark mode


### 0 - Define WSIs paths and WSI reader

In [ ]:
# Define here the folder with WSIs
# Update this path to point to your local data directory
DATA_PATH = "./data"

# 24 WSIs --> range from 1 to 25 (excluded)
img_paths = [Path(f"{DATA_PATH}/CRC_WSI/{i}.svs") for i in range(1, 25)]
annotation_paths = [Path(f"{DATA_PATH}/annotations/{i}.xml") for i in range(1, 25)]

BE CAREFUL: WSI x.svs will be addressed in the wsi_reader list as wsi_reader[x - 1]

In [ ]:
# Inizializza una lista per memorizzare i lettori di WSI
wsi_reader = []

# Carica ogni immagine nel lettore WSI
for img_path in img_paths:
    wsi_reader.append(WSIReader.open(input_img=img_path))


# Runna solo se vuoi info su tutte le WSIs
# RICORDA DI ESEGUIRE: pip installe prettytable
# -----------------------------------------------------------------------
# # Importa il modulo necessario
# from prettytable import PrettyTable

# # Cambia i parametri relativi a slide_dimensions
# info_res = 0.5
# info_unit = "mpp"

# # Crea una tabella per mostrare le informazioni
# table = PrettyTable()

# # Definisci le colonne della tabella
# table.field_names = ["Image Index", "Property", "Value"]

# # Imposta l'allineamento delle colonne a sinistra
# table.align["Property"] = "l"
# table.align["Value"] = "l"

# # Itera su ogni lettore WSI e stampa le informazioni
# for index, reader in enumerate(wsi_reader):
#     wsi_info = reader.info.as_dict()
    
#     # Aggiungi le informazioni dell'immagine alla tabella
#     for key, value in wsi_info.items():
#         table.add_row([index + 1, key, value])
    
#     # Aggiungi la dimensione della slide alla tabella
#     slide_dimensions = reader.slide_dimensions(info_res, info_unit)
#     table.add_row([index + 1, f"Slide Dimensions {info_unit} {info_unit}", slide_dimensions])
    
#     # Aggiungi una linea divisoria alla fine di ogni immagine
#     table.add_row(["-"*10, "-"*20, "-"*40])

# # Rimuovi l'ultima linea divisoria
# table.del_row(-1)

# # Stampa la tabella in un formato leggibile
# print(table)


**objective_power**: Potere dell'obiettivo del microscopio utilizzato per acquisire l'immagine WSI.
Valore: 40.0 indica che l'immagine è stata acquisita con un obiettivo a 40x.

**slide_dimensions**: Dimensioni dell'immagine WSI alla massima risoluzione (livello di base).
Valore: (100720, 76803) rappresenta la larghezza (100720 pixel) e l'altezza (76803 pixel) dell'immagine a piena risoluzione.

**level_count**: Numero di livelli di risoluzione nella piramide dell'immagine WSI.
Valore: 4 indica che l'immagine ha 4 livelli di risoluzione.

**level_dimensions**: Dimensioni dell'immagine WSI a ciascun livello di risoluzione.
Valore: ((100720, 76803), (25180, 19200), (6295, 4800), (3147, 2400)) rappresenta le dimensioni dell'immagine a ciascun livello. Il primo livello (100720, 76803) è la risoluzione massima, il secondo livello (25180, 19200) è 1/4 della risoluzione massima, e così via.

**level_downsamples**: Fattori di downsampling per ciascun livello di risoluzione.
Valore: [1.0, 4.000078125, 16.0003125, 32.00316710359072] rappresenta il fattore di riduzione della risoluzione rispetto al livello di base. 1.0 indica nessun downsampling (livello di base), 4.000078125 indica che il secondo livello è ridotto di circa 4 volte rispetto al livello di base, e così via.

**vendor**: Produttore dello scanner utilizzato per acquisire l'immagine WSI.
Valore: 'aperio' indica che l'immagine è stata acquisita con uno scanner prodotto da Aperio.

**mpp**: Micron per pixel (mpp), che indica la dimensione fisica di ciascun pixel.
Valore: (0.252, 0.252) significa che ogni pixel dell'immagine rappresenta 0.252 micron.

**axes**: Ordine degli assi dell'immagine.
Valore: 'YXS' indica che l'immagine è memorizzata con l'asse Y (altezza) come primo asse, X (larghezza) come secondo asse e S (canali) come terzo asse.

### Patch Extraction

A very common practice in computational pathology, when analysing large histology images or WSIs, is to extract overlapping patches from that image and analyse them one by one. Deep Learning models often cannot accept large images due to memory limitations.

If you want to extract patches with the highest level of detail available in your WSI, you would typically use resolution=0 and units="level", meaning you are requesting patches from the highest resolution level of the image pyramid.

### 1 - Computing all the coordinates per WSI (not taking into account tissue masks)

In [4]:
# CHANGE PARAMETERS FROM HERE
patch_size = (256,256)
res = 0.5
unit = "mpp"  #level, mpp, power, baseline

patch_extractors = []
all_coordinates_per_wsi = []

# Compute all the patches coordinates for each WSI
def calculate_patch_coordinates(wsi_reader):
    for idx, wsi in enumerate(wsi_reader):
        extractor = PatchExtractor(
            input_img=wsi,                  # (str, Path, numpy.ndarray) – Input image for patch extraction
            patch_size=patch_size,          # (int or tuple(int)) – Patch size tuple (width, height)
            resolution=res,                 # (Resolution) – Resolution at which to read the image, default = 0
            units=unit,                     # (Units) – Units of resolution, default = “level”
            pad_mode='constant',            # (str) – Method for padding at edges of the WSI. Default to ‘constant’. See numpy.pad() for more information
            pad_constant_values=0,          # (int or tuple(int)) – Values to use with constant padding. Defaults to 0. See numpy.pad() for more
            min_mask_ratio=0,               # (float) – Area in percentage that a patch needs to contain of positive mask to be included. Defaults to 0
            within_bound=False              # (bool) – Whether to extract patches beyond the input_image size limits
                                            # If False, extracted patches at margins will be padded appropriately based on pad_constant_values and pad_mode
                                            # If True, patches at the margins whose bounds would exceed the mother image dimensions would be neglected. Default is False
        )

        # Save the patch extractor for each WSI
        patch_extractors.append(extractor)

        # Now we can use the get_coordinates method of the PatchExtractor class
        coordinates_list = extractor.get_coordinates(        # RETURNS: A list of coordinates in [start_x, start_y, end_x, end_y] format to be used for patch extraction.
            patch_output_shape=patch_size,              # (tuple (int, int) or numpy.ndarray) – Specifies the output shape of requested patches to be extracted
                                                        # from mother image at desired resolution and units. This argument is also expected to be in (width, height) format.
                                                        # If this is not provided, patch_output_shape will be the same as patch_input_shape
            image_shape=wsi.info.slide_dimensions,      # (tuple (int, int) or numpy.ndarray) – This argument specifies the shape of mother image
                                                        # (the image we want to extract patches from) at requested resolution and units and it is expected to be in (width, height) format
            patch_input_shape=patch_size,               # (tuple (int, int) or numpy.ndarray) – Specifies the input shape of requested patches to be extracted
                                                        # from mother image at desired resolution and units. This argument is also expected to be in (width, height) format.
            stride_shape=patch_size,                    # NO OVERLAP WITH STRIDE = PATCH_SIZE
                                                        # (tuple (int, int) or numpy.ndarray) – The stride that is used to calculate the patch location during the patch extraction.
                                                        # If patch_output_shape is provided, next stride location will base on the output rather than the input.
            input_within_bound=False                    # (bool) – Whether to include the patches where their input location exceed the margins of mother image.
                                                        # If True, the patches with input location exceeds the image_shape would be neglected.
                                                        # Otherwise, those patches would be extracted with Reader function and appropriate padding.
        )

        # Verify the content of coordinates_list
        # if get_coordinates returns 2 identical arrays, keep only one of them
        if len(coordinates_list) == 2 and np.array_equal(coordinates_list[0], coordinates_list[1]):
            coordinates = coordinates_list[0]
        else:
            coordinates = coordinates_list

        # Saving all the coordinate for each WSI
        all_coordinates_per_wsi.append(coordinates)
        
        # Runna solo se vuoi info
        # --------------------------------------
        # # Summary of computed coordinates
        # info = wsi.slide_dimensions(resolution=0, units="level")
        # print(f"WSI {idx + 1}.svs con dimensioni originali: {info}")
        # info = wsi.slide_dimensions(resolution=res, units=unit)
        # print(f"WSI {idx + 1}.svs con dimensioni (dipendenti da resolution {res} e unit '{unit}'): {info}")
        # print(f"Numero di patch:\n {len(coordinates)}")
        # print(f"Prime coordinate di patch:\n {coordinates[:2]}")
        # print(f"Ultime coordinate di patch:\n {coordinates[-2:]}\n")
    
    return all_coordinates_per_wsi

# Calling the function to compute all the coordinates for each patch of each WSI
# all_coordinates_per_wsi[0] --> coordinates of the first WSI in the format [start_x, start_y, end_x, end_y]
all_coordinates_per_wsi = calculate_patch_coordinates(wsi_reader)



### 2 - Creating an Otsu mask for each WSI

In [ ]:
# List for Otsu masks of each WSI
otsu_masks = []

# Parameters for the creation of each Otsu mask
mask_res = 1.25
mask_unit = "power"

# We are using WSIreader.tissue_mask from TiaToolbox.
# It returns a WSIVirtualReader which is needed to call filter_coordinates later

# Function to create Otsu mask with desired resolution and unit
def create_otsu_mask(wsi_reader, resolution, units):
    otsu_mask_reader = wsi_reader.tissue_mask(method='otsu', resolution=resolution, units=units)
    return otsu_mask_reader

# Iterate on each WSI and append mask in otsu_masks
for idx, wsi in enumerate(wsi_reader):
    otsu_mask = create_otsu_mask(wsi, resolution=mask_res, units=mask_unit)
    otsu_masks.append(otsu_mask)
    print(f"Maschera Otsu creata per WSI {idx + 1}.svs")


### 2-bis H&E Masks for WSI 8, 14, 18, 19

La soglia di Otsu è un valore di soglia calcolato dall'algoritmo di Otsu, utilizzato per convertire un'immagine in una maschera binaria. La maschera di Otsu è l'immagine risultante dopo aver applicato questa soglia, dove i pixel al di sopra della soglia sono considerati "tessuto" e quelli al di sotto sono "non tessuto".

Quindi, la soglia di Otsu è il valore di soglia calcolato, mentre la maschera di Otsu è l'immagine binaria ottenuta applicando questa soglia.

In [6]:
# Tratto dal paper di cui sotto nella sezione LICENCE
"""Code for H&E Otsu Thresholding."""

# from typing import Union, Tuple
# import numpy as np
# from skimage.filters import threshold_otsu
# from scipy.signal import convolve2d

def he_otsu(
    image: np.ndarray,
    blur_kernel_width: int = 0,
    nbins: int = 256,
    hist: Union[np.ndarray, Tuple[np.ndarray, np.ndarray]] = None
) -> Tuple[np.ndarray, float]:
    """Calculates the H&E Otsu threshold.

    Args:
        image (np.ndarray): An RGB image
        blur_kernel_width (int): Blurs the mask with the given blur kernel
        nbins (int, optional): Number of bins used to calculate histogram.
                            This value is ignored for integer arrays.
                            Defaults to 256.
        hist (Union[np.ndarray, Tuple[np.ndarray,np.ndarray]], optional):
                            Histogram from which to determine the threshold, and optionally a
                            corresponding array of bin center intensities. If no hist provided,
                            this function will compute it from the image. Default to None.

    Returns:
        Tuple[np.ndarray, float]: Tissue mask and upper threshold value. All pixels with an
        intensity higher than this value are assumed to be tissue.
    """

    red_channel = image[:, :, 0].astype(float)
    green_channel = image[:, :, 1].astype(float)
    blue_channel = image[:, :, 2].astype(float)

    red_to_green_mask = np.maximum(red_channel - green_channel, 0)
    blue_to_green_mask = np.maximum(blue_channel - green_channel, 0)

    tissue_heatmap = red_to_green_mask * blue_to_green_mask

    threshold = threshold_otsu(
        red_to_green_mask * blue_to_green_mask, nbins=nbins, hist=hist
    )

    mask = tissue_heatmap > threshold

    if blur_kernel_width != 0:
        blur_kernel = np.ones((blur_kernel_width, blur_kernel_width))
        mask = convolve2d(mask, blur_kernel, mode = "same")
        mask = mask > 0
        # mask = mask > 0

    return mask, threshold


LICENCE CONDITIONS

Copyright (c) 2023 Benjamin Alexander Schreiber
All rights reserved.

For details see the paper:
B.A. Schreiber et al.,
Rapid Artefact Removal and Tissue Segmentation in Haematoxylin and Eosin Stained Biopsies.
Scientific Reports (https://doi.org/10.1038/s41598-023-50183-4), 2024.

Permission to use, copy, modify, and distribute this software and its documentation for educational, research, and non-commercial purposes, without fee and without a signed licensing agreement, is hereby granted, provided that the above copyright notice and this paragraph appear in all copies, modifications, and distributions.

Any commercial use or any redistribution of this software requires a licence. For further details, contact Benjamin Alexander Schreiber (ben.a.schreiber@hotmail.com).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

In [ ]:
# Parameters for the creation of each H&E-Otsu-Threshold masks
mask_res = 1.25
mask_unit = "power"

# Lista per salvare le maschere
H_E_Otsu_Thresh_masks = []

def process_and_show_image(wsi_reader, index, resolution=mask_res, units=mask_unit):
    # Crea una thumbnail dell'immagine WSI a una risoluzione inferiore
    thumbnail = wsi_reader[index].slide_thumbnail(resolution=resolution, units=units)
    thumbnail_array = np.array(thumbnail)

    # Applica il metodo per rimuovere i segni di pennarello sulla thumbnail
    tissue_mask, threshold = he_otsu(thumbnail_array)

    # Salva la maschera nella lista
    H_E_Otsu_Thresh_masks.append(tissue_mask)

    # # Visualizza il risultato
    # plt.figure(figsize=(12, 6))
    # plt.subplot(1, 2, 1)
    # plt.imshow(thumbnail_array)
    # plt.title(f'WSI {index + 1}.svs')
    # plt.axis('off')
    
    # plt.subplot(1, 2, 2)
    # plt.imshow(tissue_mask, cmap='gray')
    # plt.title(f'H&E-Otsu-Threshold Mask (Threshold: {threshold})')
    # plt.axis('off')
    # plt.show()

# Applica il metodo alle immagini con indice 7, 13, 17 e 18
indices_to_process = [7, 13, 17, 18]
for index in indices_to_process:
    process_and_show_image(wsi_reader, index)

# Ora H_E_Otsu_Thresh_masks contiene tutte le maschere create
print("Maschere salvate con successo nelle seguenti posizioni della lista:")
for i in range(len(H_E_Otsu_Thresh_masks)):
    print(f"H_E_Otsu_Thresh_masks[{i}]: maschera relativa a {indices_to_process[i] + 1}.svs")


### 3 - Filter patches coordinates according to obtained Otsu masks

In [ ]:
### 3 - Applicare filter_coordinates usando le maschere Otsu

valid_coordinates_per_wsi = []

# Funzione per applicare filter_coordinates
def filter_patch_coordinates(patch_extractors, otsu_masks, all_coordinates_per_wsi):
    for idx, extractor in enumerate(patch_extractors):
        mask_reader = otsu_masks[idx]
        coordinates = all_coordinates_per_wsi[idx]
        
        flags = extractor.filter_coordinates(
            mask_reader=mask_reader,
            coordinates_list=coordinates,
            wsi_shape=extractor.wsi.slide_dimensions(resolution=res, units=unit),
            min_mask_ratio=0.7  # Adatta questo valore secondo le tue esigenze
        )

        # Filtra le coordinate usando le flags ottenute da filter_coordinates
        valid_coordinates = coordinates[flags]
        
        valid_coordinates_per_wsi.append(valid_coordinates)
        print(f"Coordinate valide filtrate per WSI {idx + 1}.svs: {len(valid_coordinates)} patch valide")
    
    return valid_coordinates_per_wsi

# Filtrare le coordinate di estrazione per ogni immagine usando le maschere Otsu
valid_coordinates_per_wsi = filter_patch_coordinates(patch_extractors, otsu_masks, all_coordinates_per_wsi)


### 3-bis Filter patches coordinates according to obtained H&E-Otsu-Threshold masks

In [ ]:
# Lista per salvare le maschere H&E ridimensionate
resized_HE_masks = []

# Funzione per ridimensionare la maschera
def resize_mask_to_wsi(mask, reference_wsi_reader, resolution=res, units=unit):     # res e unit sono definite all'inizio del notebook
    target_shape = reference_wsi_reader.slide_dimensions(resolution=resolution, units=units)
    resized_mask = resize(mask, (target_shape[1], target_shape[0]), preserve_range=True, anti_aliasing=False).astype(np.uint8)
    return resized_mask

# Funzione per convertire le maschere H&E-Otsu in VirtualWSIReader e salvarle
def create_virtual_wsi_reader_from_mask(mask, reference_wsi_reader):
    resized_mask = resize_mask_to_wsi(mask, reference_wsi_reader)
    virtual_wsi_reader = VirtualWSIReader(
        input_img=resized_mask,
        mode='bool'  # Poiché la maschera è binaria
    )

    resized_HE_masks.append(virtual_wsi_reader)  # Salva la maschera ridimensionata e trasformata nella lista
    
    return virtual_wsi_reader

# Funzione per applicare filter_coordinates utilizzando le maschere H&E-Otsu
def filter_patch_coordinates_with_he(patch_extractors, he_masks, all_coordinates_per_wsi, indices):
    valid_coordinates_he = []

    for idx in indices:
        extractor = patch_extractors[idx]  # Ottieni l'estrattore di patch corrispondente all'indice
        reference_wsi_reader = wsi_reader[idx]
        mask = he_masks[indices.index(idx)]  # Ottieni la maschera H&E corrispondente --> la lista delle maschere H&E ha solo 4 elementi,
                                             # che però corrispondono ad indici diversi nella lista wsi_reader, ossia gli indici salvati in indices_to_process
        virtual_wsi_reader = create_virtual_wsi_reader_from_mask(mask, reference_wsi_reader)
        coordinates = all_coordinates_per_wsi[idx]
        
        flags = extractor.filter_coordinates(
            mask_reader=virtual_wsi_reader,
            coordinates_list=coordinates,
            wsi_shape=reference_wsi_reader.slide_dimensions(resolution=res, units=unit),
            min_mask_ratio=0.7  # Adatta questo valore secondo le tue esigenze
        )

        # Filtra le coordinate usando le flags ottenute da filter_coordinates
        valid_coordinates = coordinates[flags]
        
        valid_coordinates_he.append(valid_coordinates)
        print(f"Coordinate valide filtrate per WSI {idx + 1}.svs: {len(valid_coordinates)} patch valide")
    
    return valid_coordinates_he

# Applica il metodo alle immagini con indice 7, 13, 17 e 18
indices_to_process = [7, 13, 17, 18]

# Supponiamo che `patch_extractors` e `all_coordinates_per_wsi` siano stati già calcolati per tutte le immagini
# Ora filtra le coordinate per le immagini con indice 7, 13, 17 e 18 usando le maschere H&E-Otsu
valid_coordinates_per_wsi_he = filter_patch_coordinates_with_he(patch_extractors, H_E_Otsu_Thresh_masks, all_coordinates_per_wsi, indices_to_process)


### 4 - Exploiting .xml Annotations

Questo codice è progettato per elaborare dati di annotazione provenienti da un file XML e generare una maschera per una determinata regione di una WSI. Le principali funzionalità includono:

1) Analizzare i dati di annotazione
2) Generare maschere basate su colori specifici
3) Sovrapporre queste maschere sull'immagine della diapositiva

In [10]:
# Funzione per ottenere i punti base dalle annotazioni nel file XML

def get_points_base(xml_file, colors_to_use, custom_colors=[]):
    # Crea una chiave di colore unendo tutti i colori da usare
    color_key = ''.join([k for k in colors_to_use])
    # Ottieni i punti e l'indice mappa dalle annotazioni XML
    points, map_idx = get_points_xml(xml_file, colors_to_use, custom_colors)
    # Crea poligoni dai punti ottenuti
    point_polys = [Polygon(point) for point in points]
    # Ritorna un dizionario con i punti, mappa degli indici, poligoni e STRtree per l'indicizzazione spaziale
    return {'points': points.copy(),
            'map_idx': map_idx.copy(),
            'polygons': point_polys.copy(),
            'STRtree': STRtree(point_polys)}

# Funzione per abbinare i colori di riferimento dalle annotazioni XML

def color_ref_match_xml(colors_to_use, custom_colors):
    # Definisce una mappa di riferimento dei colori con i codici esadecimali e nomi
    color_ref = [(65535, 1, 'yellow'), (65280, 2, 'green'), (255, 3, 'red'), (16711680, 4, 'blue'),
                 (16711808, 5, 'purple'), (np.nan, 6, 'other')] + custom_colors

    # Se sono specificati colori da usare
    if colors_to_use is not None:
        # Converti in minuscolo se è una stringa singola
        if isinstance(colors_to_use, str):
            colors_to_use = colors_to_use.lower()
        else:
            # Converti tutti gli elementi della lista in minuscolo
            colors_to_use = [color.lower() for color in colors_to_use]
        # Filtra la mappa dei colori per includere solo quelli specificati
        color_map = [c for c in color_ref if c[2] in colors_to_use]
    else:
        # Se nessun colore è specificato, usa tutti i colori di riferimento
        color_map = color_ref

    return color_map

# Funzione per ottenere i punti delle annotazioni dall'XML
def get_points_xml(xml_file, colors_to_use, custom_colors):
    # Abbina i colori di riferimento
    color_map = color_ref_match_xml(colors_to_use, custom_colors)
    # Crea una chiave di colore concatenando i nomi dei colori
    color_key = ''.join([k[2] for k in color_map])
    full_map = color_ref_match_xml(None, custom_colors)

    # Crea l'oggetto albero dell'elemento XML
    tree = ET.parse(xml_file)
    # Ottieni l'elemento radice
    root = tree.getroot()

    map_idx = []
    points = []

    # Cicla attraverso tutte le annotazioni nel file XML
    for annotation in root.findall('Annotation'):
        # Ottieni il colore della linea dell'annotazione
        line_color = int(annotation.get('LineColor'))
        # Trova l'indice mappato per il colore della linea
        mapped_idx = [item[1] for item in color_map if item[0] == line_color]

        # Se non c'è corrispondenza con il colore mappato e il colore "altro" è specificato
        if not mapped_idx and not [item[1] for item in full_map if item[0] == line_color]:
            if 'other' in [item[2] for item in color_map]:
                mapped_idx = [item[1] for item in color_map if item[2] == 'other']

        # Se c'è un indice mappato valido
        if mapped_idx:
            if isinstance(mapped_idx, list):
                mapped_idx = mapped_idx[0]

            # Cicla attraverso tutte le regioni dell'annotazione
            for regions in annotation.findall('Regions'):
                for annCount, region in enumerate(regions.findall('Region')):
                    map_idx.append(mapped_idx)
                    # Ottieni i vertici delle regioni
                    for vertices in region.findall('Vertices'):
                        points.append([None] * len(vertices.findall('Vertex')))
                        for k, vertex in enumerate(vertices.findall('Vertex')):
                            points[-1][k] = (int(float(vertex.get('X'))), int(float(vertex.get('Y'))))

    # Ordina i punti secondo l'ordine dei colori
    sort_order = [x[1] for x in color_map]
    new_order = []
    for x in sort_order:
        new_order.extend([index for index, v in enumerate(map_idx) if v == x])

    points = [points[x] for x in new_order]
    map_idx = [map_idx[x] for x in new_order]

    return points, map_idx

# Funzione per generare la maschera delle annotazioni
def get_annotation_mask(xml_file, slide, level_downsample, colors_to_use=None):
    # Ritorna la maschera di un tile
    coords = (0, 0)
    if 'openslide.bounds-width' in slide.properties.keys():
        # Considera solo la regione non vuota della diapositiva
        bounds_width = int(slide.properties['openslide.bounds-width'])
        bounds_height = int(slide.properties['openslide.bounds-height'])
        bounds_x = int(slide.properties['openslide.bounds-x'])
        bounds_y = int(slide.properties['openslide.bounds-y'])

        region_lv0 = (bounds_x, bounds_y, bounds_width, bounds_height)
    else:
        # Se non è disponibile la bounding box della regione non vuota della diapositiva
        region_lv0 = (0, 0, slide.level_dimensions[0][0], slide.level_dimensions[0][1])
    region_lv0 = [round(x) for x in region_lv0]
    region_lv_selected = [round(x * level_downsample) for x in region_lv0]

    # Ottieni i punti base dalle annotazioni XML
    points_dict = get_points_base(xml_file, colors_to_use)
    points = points_dict['points']
    map_idx = points_dict['map_idx']
    point_polys = points_dict['polygons']
    point_tree = points_dict['STRtree']

    # Crea un poligono per il tile
    tile_poly = Polygon([(coords[0], coords[1]),
                         (coords[0], coords[1] + region_lv0[3]),
                         (coords[0] + region_lv0[2], coords[1] + region_lv0[3]),
                         (coords[0] + region_lv0[2], coords[1])])
    # Inizializza la maschera
    mask = np.zeros((region_lv_selected[3], region_lv_selected[2]), dtype=np.uint8)

    if not point_polys:
        point_polys = [Polygon(point) for point in points]
        point_tree = STRtree(point_polys)

    index_by_id = dict((id(pt), i) for i, pt in enumerate(point_polys))

    # Trova i punti che intersecano con il poligono del tile
    intersecting_points = [index_by_id[id(point_tree.geometries[pt])] for pt in point_tree.query(tile_poly)]

    points_maps = [point_map for idx, point_map in enumerate(zip(points, map_idx)) if idx in intersecting_points]
    if points_maps:
        points, map_idx = zip(*points_maps)
        points = [[(int(p[0] * level_downsample), int(p[1] * level_downsample)) for p in pointSet] for pointSet in points]
        coords = tuple([int(c * level_downsample) for c in coords])
        points = [[(int(p[0] - coords[0]), int(p[1] - coords[1])) for p in pointSet] for pointSet in points]
        for annCount, pointSet in enumerate(points):
            cv2.fillPoly(mask, [np.asarray(pointSet).reshape((-1, 1, 2))], map_idx[annCount])
    mask[mask != 0] = 255
    return mask


In [ ]:
# Lista in cui vengono salvate le maschere estratte dalle annotazioni .xml
annotationmasks = []

for i in range(len(img_paths)):
    wsi_file = img_paths[i]
    xml_file = annotation_paths[i]

    # Open the WSI file with OpenSlide library --> annotations masks are created with OpenSlide library
    slide = openslide.OpenSlide(wsi_file)

    original_dimensions = wsi_reader[i].slide_dimensions(resolution=0, units="level")
    working_dimensions = wsi_reader[i].slide_dimensions(resolution=res, units=unit)

    x_original = original_dimensions[0]
    x_final = working_dimensions[0]

    print(f"Original dimensions of WSI {i + 1}.svs: {original_dimensions}")
    print(f"Actual working dimensions of WSI {i + 1}.svs: {working_dimensions}")

    # il level downsample deve essere calcolato per ogni immagine, in base alla resolution ed unità di misura con cui lavoriamo
    # Level_downsample: in realtà quello che fa è soltanto un totpixel * 0.5 su entrambi gli assi
    # il calcolo si fa facendo pixel_scalati/pixel_originali, pixel_scalati sono i pixel dopo essere stati
    # convertiti a 0.5mpp

    downsample_factor = x_final / x_original

    annotationmask = get_annotation_mask(xml_file=xml_file, slide=slide, level_downsample=downsample_factor, colors_to_use=['red'])
    print(f"Annotation mask shape (x and y inverted): {annotationmask.shape[1]} {annotationmask.shape[0]}")
    print("--------------------------------------------")

    annotationmasks.append(annotationmask)


### Stain Normalization

In [12]:
target_image = data.stain_norm_target()
stain_normalizer = stainnorm.get_normalizer("Macenko")

stain_normalizer.fit(target_image)

### FOLDER CHECK AND CREATION !

In [36]:
# Creare le directory "tumorali", "non tumorali" e "no_annotations" nella cartella "src/data/patches"
base_dir = "src/data/patches"
tumor_dir = os.path.join(base_dir, "tumorali")
non_tumor_dir = os.path.join(base_dir, "non_tumorali")
no_annotation_dir = os.path.join(base_dir, "no_annotations")

# Cancella e ricrea le directory se esistono già
if os.path.exists(tumor_dir):
    shutil.rmtree(tumor_dir)
if os.path.exists(non_tumor_dir):
    shutil.rmtree(non_tumor_dir)
if os.path.exists(no_annotation_dir):
    shutil.rmtree(no_annotation_dir)

os.makedirs(tumor_dir, exist_ok=True)
os.makedirs(non_tumor_dir, exist_ok=True)
os.makedirs(no_annotation_dir, exist_ok=True)

### 5 - Patches extraction according to valid coordinates and classified as tumoral (folder "tumorali") or not

In [ ]:
# Funzione per calcolare le coordinate delle patch tumorali per una singola immagine
def calculate_cancer_coords_for_single_image(mask, valid_coordinates, patch_size, idx):
    coords = []

    for coord in valid_coordinates:
        start_x, start_y, end_x, end_y = coord

        # Estrarre la patch dalla maschera
        mask_patch = mask[start_y:end_y, start_x:start_x + patch_size[0]]

        # Calcolare la percentuale di pixel non-zero nella patch della maschera
        non_zero_pixels = np.count_nonzero(mask_patch)
        total_pixels = mask_patch.size
        non_zero_percentage = (non_zero_pixels / total_pixels) * 100

        # Controllare se almeno il 51% dei pixel è non-zero
        if non_zero_percentage >= 51:
            coords.append([start_x, start_y, end_x, end_y])
        
    print(f"Coordinate delle patch tumorali calcolate con successo per {idx + 1}.svs")
    
    return coords

# Lista per salvare le coordinate delle patch tumorali per tutte le immagini
cancer_coords = [[] for _ in range(len(wsi_reader))]

# Esegui il calcolo delle coordinate delle patch tumorali per ogni immagine
for idx in range(len(wsi_reader)):
    cancer_coords[idx] = calculate_cancer_coords_for_single_image(annotationmasks[idx], valid_coordinates_per_wsi[idx], patch_size, idx) # patch_size è definita all'inizio del notebook

print("Coordinate delle patch tumorali calcolate con successo.")


In [ ]:
# Funzione per salvare le patch nelle directory appropriate
def save_patches(wsi_reader, valid_coordinates, tumor_coordinates, idx, tumor_dir=tumor_dir, non_tumor_dir=non_tumor_dir):
    # Convertiamo tumor_coordinates in un set di tuple per confronto rapido
    tumor_coordinates_set = set(map(tuple, tumor_coordinates))

    for i, coord in enumerate(valid_coordinates):
        start_x, start_y, end_x, end_y = coord

        # Leggiamo la giusta patch con le coordinate corrette
        patch_array = wsi_reader.read_bounds((start_x, start_y, end_x, end_y), resolution=res, units=unit, coord_space="resolution")

        # Stain normalization
        patch_array_normalized = stain_normalizer.transform(patch_array.copy())

        patch = Image.fromarray(patch_array_normalized)

        # Determina la directory di salvataggio
        if tuple(coord) in tumor_coordinates_set:
            patch_path = os.path.join(tumor_dir, f"patch_{idx + 1}_{i}.png")
        else:
            patch_path = os.path.join(non_tumor_dir, f"patch_{idx + 1}_{i}.png")
        
        patch.save(patch_path)

# Indici delle immagini da saltare
skip_indices = {7, 13, 17, 18, 20}

# Esegui il salvataggio delle patch per ogni immagine
for idx in range(24):
    if idx in skip_indices:
        continue
    save_patches(wsi_reader[idx], valid_coordinates_per_wsi[idx], cancer_coords[idx], idx)

print(f"Patch salvate con successo per tutte le immagini, escluse quelle con indice {skip_indices}")


### 5bis - Patches extraction according to valid coordinates related to H&E-Otsu-Threshold masks, and classified as tumoral (folder "tumorali") or not

In [ ]:
# Funzione per calcolare le coordinate delle patch tumorali per una singola immagine usando maschere H&E-Otsu
def calculate_cancer_coords_for_single_image_he(mask, valid_coordinates, patch_size, idx):
    coords = []

    for coord in valid_coordinates:
        start_x, start_y, end_x, end_y = coord

        # Estrarre la patch dalla maschera
        mask_patch = mask[start_y:end_y, start_x:start_x + patch_size[0]]

        # Calcolare la percentuale di pixel non-zero nella patch della maschera
        non_zero_pixels = np.count_nonzero(mask_patch)
        total_pixels = mask_patch.size
        non_zero_percentage = (non_zero_pixels / total_pixels) * 100

        # Controllare se almeno il 51% dei pixel è non-zero
        if non_zero_percentage >= 51:
            coords.append([start_x, start_y, end_x, end_y])
        
    print(f"Coordinate delle patch tumorali calcolate con successo per {idx + 1}.svs")
    
    return coords

# Lista per salvare le coordinate delle patch tumorali per le immagini con maschere H&E-Otsu
cancer_coords_he = [[] for _ in range(4)]  # solo 4 immagini

# Indici delle immagini con segni di pennarello
indices_to_process = [7, 13, 17, 18]

# Esegui il calcolo delle coordinate delle patch tumorali per ogni immagine con maschera H&E-Otsu
for i, idx in enumerate(indices_to_process):
    cancer_coords_he[i] = calculate_cancer_coords_for_single_image_he(annotationmasks[idx], valid_coordinates_per_wsi_he[i], patch_size, idx) # patch_size è definita all'inizio del notebook

print("Coordinate delle patch tumorali calcolate con successo per le immagini con maschere H&E-Otsu.")

In [ ]:
# Funzione per salvare le patch nelle directory appropriate
def save_patches_he(wsi_reader, valid_coordinates, tumor_coordinates, idx, tumor_dir=tumor_dir, non_tumor_dir=non_tumor_dir):
    # Convertiamo tumor_coordinates in un set di tuple per confronto rapido
    tumor_coordinates_set = set(map(tuple, tumor_coordinates))

    for i, coord in enumerate(valid_coordinates):
        start_x, start_y, end_x, end_y = coord

        # Leggiamo la giusta patch con le coordinate corrette
        patch_array = wsi_reader.read_bounds((start_x, start_y, end_x, end_y), resolution=res, units=unit, coord_space="resolution")

        # Stain normalization
        patch_array_normalized = stain_normalizer.transform(patch_array.copy())

        patch = Image.fromarray(patch_array_normalized)

        # Determina la directory di salvataggio
        if tuple(coord) in tumor_coordinates_set:
            patch_path = os.path.join(tumor_dir, f"patch_{idx + 1}_{i}.png")
        else:
            patch_path = os.path.join(non_tumor_dir, f"patch_{idx + 1}_{i}.png")
        
        patch.save(patch_path)
    
    print(f"Patches salvate con successo per {idx + 1}.svs")

# Indici delle immagini con segni di pennarello
indices_to_process = [7, 13, 17, 18]

# Esegui il salvataggio delle patch per ogni immagine con maschera H&E-Otsu
for i, idx in enumerate(indices_to_process):
    save_patches_he(wsi_reader[idx], valid_coordinates_per_wsi_he[i], cancer_coords_he[i], idx)

print("Patches salvate con successo per le immagini con maschere H&E-Otsu.")


### 5tris - Patches extraction according to valid coordinates but no annotation.xml is present

In [ ]:
# Funzione per salvare le patch dalla immagine con indice 20 nella directory "no_annotations"
def save_patches_no_annotations(wsi_reader, valid_coordinates, idx, no_annotation_dir=no_annotation_dir):
    for i, coord in enumerate(valid_coordinates):
        start_x, start_y, end_x, end_y = coord

        # Leggiamo la giusta patch con le coordinate corrette
        patch_array = wsi_reader.read_bounds((start_x, start_y, end_x, end_y), resolution=res, units=unit, coord_space="resolution")

        # Stain normalization
        patch_array_normalized = stain_normalizer.transform(patch_array.copy())

        patch = Image.fromarray(patch_array_normalized)

        # Salva nella directory no_annotations
        patch_path = os.path.join(no_annotation_dir, f"patch_{idx + 1}_{i}.png")
        patch.save(patch_path)

# Salva le patch per l'immagine con indice 20 nella directory "no_annotations"
save_patches_no_annotations(wsi_reader[20], valid_coordinates_per_wsi[20], 20)

print("Patch salvate con successo per l'immagine con indice 20 nella directory no_annotations.")

------------------------------------------------------------------------------------------------

### PLOT SECTION

In [14]:
def plot_valid_coordinates_only(valid_coordinates, title):
    """
    Plotta solo le coordinate valide.

    Parameters:
    - valid_coordinates: numpy.ndarray, le coordinate valide nella forma [start_x, start_y, end_x, end_y].
    - title: str, titolo del grafico.
    """
    # Crea la figura e gli assi
    fig, ax = plt.subplots(1, figsize=(12, 12))
    
    # Aggiungi i rettangoli per ogni coordinata valida
    for coord in valid_coordinates:
        start_x, start_y, end_x, end_y = coord
        width = end_x - start_x
        height = end_y - start_y
        rect = patches.Rectangle((start_x, start_y), width, height, linewidth=1, edgecolor='r', facecolor='none')
        ax.add_patch(rect)
    
    # Imposta i limiti degli assi in base alle coordinate
    ax.set_xlim(valid_coordinates[:, 0].min(), valid_coordinates[:, 2].max())
    ax.set_ylim(valid_coordinates[:, 1].min(), valid_coordinates[:, 3].max())
    
    # Inverti l'asse y per allineare con le coordinate dell'immagine
    ax.invert_yaxis()
    
    # Titolo e mostra la trama
    ax.set_title(title)
    plt.show()


In [ ]:
# Plotta una thumbnail dellla WSI e sovrappone la sua maschera delle annotazioni .xml

def overlap(slide, mask, level_downsample, colormap=cm.get_cmap('Blues'), alpha=0.5):
    if 'openslide.bounds-width' in slide.properties.keys():
        bounds_width = int(slide.properties['openslide.bounds-width'])
        bounds_height = int(slide.properties['openslide.bounds-height'])
        bounds_x = int(slide.properties['openslide.bounds-x'])
        bounds_y = int(slide.properties['openslide.bounds-y'])
        region_lv0 = (bounds_x, bounds_y, bounds_width, bounds_height)
    else:
        region_lv0 = (0, 0, slide.level_dimensions[0][0], slide.level_dimensions[0][1])

    region_lv0 = [round(x) for x in region_lv0]
    region_lv_selected = [round(x * level_downsample) for x in region_lv0]
    
    # Get a downsampled version of the slide
    slide_image = slide.get_thumbnail((region_lv_selected[2], region_lv_selected[3]))
    
    # Downsample the mask to the same size as the slide thumbnail
    mask_downsampled = cv2.resize(mask, (region_lv_selected[2], region_lv_selected[3]), interpolation=cv2.INTER_NEAREST)
    
    # Apply the colormap to the mask
    map_ = colormap(mask_downsampled / mask_downsampled.max())
    
    # Convert to a PIL image
    roi_map = Image.fromarray((map_ * 255).astype('uint8'))
    roi_map.putalpha(int(255 * alpha))  # Set transparency
    
    # Convert the slide image to RGBA
    slide_image = slide_image.convert('RGBA')
    
    # Composite the images
    slide_image.alpha_composite(roi_map)
    slide_image = slide_image.convert('RGBA')
    
    return slide_image

In [16]:
def show_side_by_side(image_1, image_2, title_1='Image 1', title_2='Image 2'):
    """
    Mostra due immagini affiancate per il confronto.

    Parameters:
    - image_1: numpy.ndarray, la prima immagine da visualizzare.
    - image_2: numpy.ndarray, la seconda immagine da visualizzare.
    - title_1: str, il titolo per la prima immagine (default='Image 1').
    - title_2: str, il titolo per la seconda immagine (default='Image 2').
    """
    plt.figure(figsize=(12, 6))

    # Mostra la prima immagine
    plt.subplot(1, 2, 1)
    plt.imshow(image_1)
    plt.title(title_1)
    plt.axis('off')

    # Mostra la seconda immagine
    plt.subplot(1, 2, 2)
    plt.imshow(image_2, cmap='gray')
    plt.title(title_2)
    plt.axis('off')

    plt.show()

In [17]:
def resize_image_in_blocks(image, scale_factor, block_size=1024):
    """
    Ridimensiona un'immagine in blocchi per evitare errori di memoria.

    Parameters:
    - image: numpy.ndarray, l'immagine da ridimensionare.
    - scale_factor: float, il fattore di scala per il ridimensionamento.
    - block_size: int, la dimensione dei blocchi da utilizzare per il ridimensionamento.

    Returns:
    - resized_image: numpy.ndarray, l'immagine ridimensionata.
    """
    target_shape = (int(image.shape[0] * scale_factor), int(image.shape[1] * scale_factor))
    resized_image = np.zeros(target_shape, dtype=np.uint8)

    for y in range(0, image.shape[0], block_size):
        for x in range(0, image.shape[1], block_size):
            block = image[y:y + block_size, x:x + block_size]
            resized_block = resize(block, (int(block.shape[0] * scale_factor), int(block.shape[1] * scale_factor)), preserve_range=True, anti_aliasing=False).astype(np.uint8)
            resized_image[int(y * scale_factor):int((y + block.shape[0]) * scale_factor), int(x * scale_factor):int((x + block.shape[1]) * scale_factor)] = resized_block

    return resized_image


def plot_mask_and_coordinates(mask, valid_coordinates, title, color, max_coords=1000, scale_factor=0.25):
    """
    Plotta la maschera e le coordinate valide sovrapposte.

    Parameters:
    - mask: numpy.ndarray, la maschera in bianco e nero.
    - valid_coordinates: list of lists, le coordinate valide nella forma [start_x, start_y, end_x, end_y].
    - title: str, titolo del grafico.
    - color: str, colore dei rettangoli.
    - max_coords: int, numero massimo di coordinate valide da plottare (default=1000).
    - scale_factor: float, fattore di scala per ridurre la risoluzione della maschera per il plotting (default=0.25).
    """
    # Riduci la risoluzione della maschera per il plotting utilizzando il ridimensionamento basato su blocchi
    mask_resized = resize_image_in_blocks(mask, scale_factor)

    # Scala le coordinate valide
    valid_coordinates_scaled = [[int(coord[0] * scale_factor), int(coord[1] * scale_factor), int(coord[2] * scale_factor), int(coord[3] * scale_factor)] for coord in valid_coordinates]

    # Seleziona un sottoinsieme di coordinate valide se necessario
    if len(valid_coordinates_scaled) > max_coords:
        valid_coordinates_scaled = valid_coordinates_scaled[:max_coords]

    # Crea la figura e gli assi
    fig, ax = plt.subplots(1, figsize=(12, 12))

    # Mostra la maschera ridimensionata
    ax.imshow(mask_resized, cmap='gray')

    # Aggiungi i rettangoli per ogni coordinata valida
    for coord in valid_coordinates_scaled:
        start_x, start_y, end_x, end_y = coord
        width = end_x - start_x
        height = end_y - start_y
        rect = patches.Rectangle((start_x, start_y), width, height, linewidth=1, edgecolor=color, facecolor='none')
        ax.add_patch(rect)

    # Imposta i limiti degli assi in base alla maschera ridimensionata
    ax.set_xlim(0, mask_resized.shape[1])
    ax.set_ylim(0, mask_resized.shape[0])

    # Inverti l'asse y per allineare con le coordinate dell'immagine
    ax.invert_yaxis()

    # Titolo e mostra la trama
    ax.set_title(title)
    plt.show()


In [18]:
# Funzione per plottare la maschera con le annotazioni .xml ridimensionata
def plot_resized_mask(mask, scale_factor=0.25):
    """
    Plotta la maschera ridimensionata.

    Parameters:
    - mask: numpy.ndarray, la maschera in bianco e nero.
    - scale_factor: float, fattore di scala per ridurre la risoluzione della maschera per il plotting (default=0.25).
    """
    # Riduci la risoluzione della maschera per il plotting
    mask_resized = resize_image_in_blocks(mask, scale_factor)

    # Plotta la maschera ridimensionata
    plt.imshow(mask_resized, cmap='gray')
    plt.title("Grayscale Image")
    plt.colorbar()  # Show color scale
    plt.show()

In [19]:
def show_wsi_and_he_mask(wsi_reader, H_E_Otsu_Thresh_masks, indices_to_process, idx):
    """
    Mostra una thumbnail della WSI originale e la sua maschera H&E-Otsu-Threshold.

    Parameters:
    - wsi_reader: lista di lettori WSI.
    - H_E_Otsu_Thresh_masks: lista di maschere H&E-Otsu-Threshold.
    - indices_to_process: lista di indici delle immagini da processare.
    - idx: indice dell'immagine da visualizzare (dentro indices_to_process).
    """
    # Ottieni l'indice della WSI originale e la maschera relativa
    wsi_idx = indices_to_process[idx]
    wsi_thumb = wsi_reader[wsi_idx].slide_thumbnail(resolution=mask_res, units=mask_unit)
    mask_thumb = H_E_Otsu_Thresh_masks[idx]

    # Visualizza la thumbnail della WSI e la maschera
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(wsi_thumb)
    plt.title(f'WSI {wsi_idx + 1}.svs')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(mask_thumb, cmap='gray')
    plt.title(f'H&E-Otsu-Threshold Mask for WSI {wsi_idx + 1}.svs')
    plt.axis('off')
    plt.show()

In [20]:
# Indice dell'immagine WSI che stiamo considerando
# ATTENZIONE: per riferirti all'immagine 1.svs, dovrai utilizzare l'indice 0
# ricorda quindi di scalare -1 al nome della foto (4.svs --> idx = 3)
idx = 0

In [ ]:
# Plotta solo le coordinate valide
plot_valid_coordinates_only(valid_coordinates_per_wsi[idx], f"Valid Coordinates of {idx + 1}.svs")

In [ ]:
# Plotta solo le coordinate valide delle immagini con maschera H&E-Otsu-Threshold
# Applica il metodo alle immagini con indice 7, 13, 17 e 18
indices_to_process = [7, 13, 17, 18]
for idx in indices_to_process:
    plot_valid_coordinates_only(valid_coordinates_per_wsi_he[indices_to_process.index(idx)], f"Valid Coordinates of {idx + 1}.svs")

In [ ]:
idx = 1
# Plotta la maschera delle annotazioni .xml e le coordinate valide sovrapposte
plot_mask_and_coordinates(annotationmasks[idx], valid_coordinates_per_wsi[idx], f"Annotation mask and Valid Coordinates of {idx + 1}.svs", "red", max_coords=20000, scale_factor=0.25) # usare 0.25 oppure 1

In [ ]:
# Applica il metodo alle immagini con indice 7, 13, 17 e 18
indices_to_process = [7, 13, 17, 18]
for idx in indices_to_process:
    # Plotta la maschera delle annotazioni .xml e le coordinate valide sovrapposte
    plot_mask_and_coordinates(annotationmasks[idx], valid_coordinates_per_wsi_he[indices_to_process.index(idx)], f"Annotation mask and Valid Coordinates of {idx + 1}.svs", "red", max_coords=20000, scale_factor=0.25)

In [ ]:
# Plotta la maschera delle annotazioni .xml e le coordinate delle patches che abbiamo considerato tumorali
plot_mask_and_coordinates(annotationmasks[idx], cancer_coords[idx], f"Annotation mask and cancer patches of {idx + 1}.svs", "blue", max_coords=20000, scale_factor=0.25)

In [ ]:
# Plotta la maschera delle annotazioni .xml e le coordinate delle patches che abbiamo considerato tumorali
plot_mask_and_coordinates(annotationmasks[idx], cancer_coords[idx], f"Annotation mask and cancer patches of {idx + 1}.svs", "blue", max_coords=20000, scale_factor=0.25)

In [ ]:
# Plotta la maschera delle annotazioni .xml e le coordinate delle patches che abbiamo considerato tumorali
indices_to_process = [7, 13, 17, 18]
for i, idx in enumerate(indices_to_process):
    plot_mask_and_coordinates(annotationmasks[idx], cancer_coords_he[i], f"Annotation mask and cancer patches of {idx + 1}.svs", "blue", max_coords=20000, scale_factor=0.25)

In [ ]:
# Showing the WSI thumbnail and its Otsu mask
for idx, wsi in enumerate(wsi_reader):
    wsi_thumb = wsi_reader[idx].slide_thumbnail(resolution=mask_res, units=mask_unit)
    mask_thumb = otsu_masks[idx].slide_thumbnail(resolution=mask_res, units=mask_unit)
    show_side_by_side(wsi_thumb, mask_thumb)

In [ ]:
# Showing the WSI thumbnail and its H&E-Otsu-Threshold mask
# Applica il metodo alle immagini con indice 7, 13, 17 e 18
indices_to_process = [7, 13, 17, 18]
for i in range(len(indices_to_process)):
    show_wsi_and_he_mask(wsi_reader, H_E_Otsu_Thresh_masks, indices_to_process, i)

In [ ]:
# Plot only the mask from annotations .xml
plot_resized_mask(annotationmasks[idx], scale_factor=0.25)

# # Plotta la maschera con scale_factor=1 --> usa questo se vuoi la maschera a dimensioni originale, esegue in meno tempo rispetto al resize
# plt.imshow(mask_resized, cmap='gray')
# plt.title("Grayscale Image")
# plt.colorbar()  # Show color scale
# plt.show()

In [ ]:
# Sovrapposizione della maschera sull'immagine della WSI
slide = openslide.OpenSlide(img_paths[idx])
mask = annotationmasks[idx]

# Imposta un valore di downsample maggiore per ridurre l'utilizzo di memoria
level_downsample = 0.05

overlap_image = overlap(slide, mask, level_downsample)

# Mostra l'immagine sovrapposta
plt.imshow(overlap_image)
plt.title(f"Overlay of Mask on WSI {idx + 1}")
plt.axis("off")
plt.show()
